<a href="https://colab.research.google.com/github/MinglesTsoi/HKSRPA/blob/main/Mark_Six_Prediction_with_DAY_Approach_v1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# 從 Google Drive 載入資料

from google.colab import drive
import os

drive.mount("/content/drive")

project_dir = "/content/drive/MyDrive/Mark_Six"
csv_path = f"{project_dir}/Mark_Six_Prediction-ALL.csv"
output_dir = f"{project_dir}/outputs"

os.makedirs(output_dir, exist_ok=True)

print("Project directory:", project_dir)
print("CSV path:", csv_path)
print("Output directory:", output_dir)
print("CSV exists:", os.path.exists(csv_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/Mark_Six
CSV path: /content/drive/MyDrive/Mark_Six/Mark_Six_Prediction-ALL.csv
Output directory: /content/drive/MyDrive/Mark_Six/outputs
CSV exists: True


In [23]:
# 讀取資料

import os
import pandas as pd

number_columns = [
    "Number1",
    "Number2",
    "Number3",
    "Number4",
    "Number5",
    "Number6",
    "Special_Number",
]


def load_mark_six_data(csv_path):
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f"找不到CSV：{csv_path}"
        )

    raw_df = pd.read_csv(
        csv_path,
        dtype={
            "Lot": str,
            "Date": str,
        },
    )

    raw_df.columns = (
        raw_df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    required_columns = (
        ["Lot", "Date"]
        + number_columns
    )

    missing_columns = [
        column_name
        for column_name in required_columns
        if column_name not in raw_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "CSV缺少欄位："
            + ", ".join(missing_columns)
        )

    if raw_df.empty:
        raise ValueError(
            "CSV沒有任何資料列。"
        )

    raw_df["Lot"] = (
        raw_df["Lot"]
        .astype("string")
        .str.strip()
    )

    raw_df["Date"] = (
        raw_df["Date"]
        .astype("string")
        .str.strip()
    )

    raw_df[number_columns] = (
        raw_df[number_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    parsed_dates = pd.to_datetime(
        raw_df["Date"],
        dayfirst=True,
        errors="coerce",
    )

    last_row_position = len(raw_df) - 1

    next_draw_row = raw_df.iloc[
        last_row_position
    ].copy()

    next_draw_number_values = (
        next_draw_row[number_columns]
    )

    is_last_row_blank_draw = (
        next_draw_number_values
        .isna()
        .all()
    )

    has_next_lot = pd.notna(
        next_draw_row["Lot"]
    ) and str(
        next_draw_row["Lot"]
    ).strip() != ""

    has_next_date = pd.notna(
        parsed_dates.iloc[
            last_row_position
        ]
    )

    if not is_last_row_blank_draw:
        raise ValueError(
            "CSV最後一列不是空白排期列。"
            "最後一列的七個號碼欄必須全部留空。"
        )

    if not has_next_lot:
        raise ValueError(
            "CSV最後一列缺少下一期期數。"
        )

    if not has_next_date:
        raise ValueError(
            "CSV最後一列缺少有效的下一期日期。"
        )

    next_lot = str(
        next_draw_row["Lot"]
    ).strip()

    next_draw_date = (
        parsed_dates.iloc[
            last_row_position
        ]
        .strftime("%Y-%m-%d")
    )

    historical_df = raw_df.iloc[
        :last_row_position
    ].copy()

    historical_df["Date"] = (
        parsed_dates.iloc[
            :last_row_position
        ].values
    )

    completed_mask = (
        historical_df[number_columns]
        .notna()
        .all(axis=1)
    )

    valid_range_mask = (
        historical_df[number_columns]
        .ge(1)
        .all(axis=1)
        & historical_df[number_columns]
        .le(49)
        .all(axis=1)
    )

    invalid_completed_df = (
        historical_df[
            completed_mask
            & ~valid_range_mask
        ]
        .copy()
    )

    if not invalid_completed_df.empty:
        print(
            "⚠️ 以下完整紀錄包含1至49以外的號碼，"
            "暫不納入模型："
        )

        display(
            invalid_completed_df[
                ["Lot", "Date"]
                + number_columns
            ]
        )

    completed_draw_df = (
        historical_df[
            completed_mask
            & valid_range_mask
            & historical_df["Date"].notna()
        ]
        .sort_values(
            ["Date", "Lot"]
        )
        .drop_duplicates(
            subset=["Lot"],
            keep="last",
        )
        .reset_index(drop=True)
        .copy()
    )

    completed_draw_df[
        number_columns
    ] = completed_draw_df[
        number_columns
    ].astype(int)

    completed_draw_df[
        "draw_set"
    ] = completed_draw_df[
        number_columns
    ].apply(
        lambda row: set(
            row.astype(int).tolist()
        ),
        axis=1,
    )

    completed_draw_df[
        "Draw_Set"
    ] = completed_draw_df[
        "draw_set"
    ]

    schedule_df = pd.DataFrame([
        {
            "Lot": next_lot,
            "Date": next_draw_date,
            "source_row":
                last_row_position,
            "number_fields_blank": True,
        }
    ])

    return {
        "raw_df": raw_df,
        "completed_draw_df":
            completed_draw_df,
        "schedule_df": schedule_df,
        "next_lot": next_lot,
        "next_draw_date":
            next_draw_date,
        "history_draw_sets":
            completed_draw_df[
                "draw_set"
            ].tolist(),
    }

In [24]:
# 路徑設定

import os
from google.colab import drive


drive.mount(
    "/content/drive",
    force_remount=False,
)


csv_path = (
    "/content/drive/MyDrive/"
    "Mark_Six/"
    "Mark_Six_Prediction-ALL.csv"
)


print(
    "CSV存在：",
    os.path.isfile(csv_path),
)

print(
    "CSV路徑：",
    csv_path,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV存在： True
CSV路徑： /content/drive/MyDrive/Mark_Six/Mark_Six_Prediction-ALL.csv


In [25]:
# 鎖定下一期

mark_six_data = load_mark_six_data(
    csv_path
)

raw_df = mark_six_data[
    "raw_df"
]

completed_draw_df = mark_six_data[
    "completed_draw_df"
]

schedule_df = mark_six_data[
    "schedule_df"
]

history_draw_sets = mark_six_data[
    "history_draw_sets"
]

next_lot = mark_six_data[
    "next_lot"
]

next_draw_date = mark_six_data[
    "next_draw_date"
]


print("=" * 60)
print("CSV排期檢查")
print("=" * 60)

print(
    "完整攪珠期數：",
    len(completed_draw_df),
)

print(
    "最後完成期數：",
    completed_draw_df.iloc[-1][
        "Lot"
    ],
)

print(
    "最後完成日期：",
    completed_draw_df.iloc[-1][
        "Date"
    ].strftime("%Y-%m-%d"),
)

print(
    "最後完成七碼：",
    sorted(
        completed_draw_df.iloc[-1][
            "draw_set"
        ]
    ),
)

print(
    "下一期：",
    next_lot,
)

print(
    "下一期日期：",
    next_draw_date,
)

print("\nCSV最後一列排期：")

display(schedule_df)

CSV排期檢查
完整攪珠期數： 928
最後完成期數： 26/101
最後完成日期： 2026-09-17
最後完成七碼： [4, 8, 17, 31, 41, 47, 48]
下一期： 26/102
下一期日期： 2026-09-19

CSV最後一列排期：


,Lot,Date,source_row,number_fields_blank
0,26/102,2026-09-19,929,True


In [26]:
# 建立各項分數

def calculate_recent_scores(
    history_draw_sets,
):
    raw_scores = {
        number: 0.0
        for number in range(1, 50)
    }

    window_settings = [
        (10, 0.50),
        (20, 0.30),
        (30, 0.20),
    ]

    for window_size, window_weight in window_settings:
        recent_draws = history_draw_sets[
            -window_size:
        ]

        frequency_counts = Counter(
            number
            for draw_set in recent_draws
            for number in draw_set
        )

        actual_window_size = len(recent_draws)

        for number in range(1, 50):
            raw_scores[number] += (
                window_weight
                * frequency_counts[number]
                / actual_window_size
            )

    return normalize_scores(raw_scores)


def calculate_gap_scores(
    history_draw_sets,
):
    raw_scores = {}

    for number in range(1, 50):
        occurrence_indices = [
            index
            for index, draw_set
            in enumerate(history_draw_sets)
            if number in draw_set
        ]

        if len(occurrence_indices) < 2:
            raw_scores[number] = 0.0
            continue

        recurrence_intervals = np.diff(
            occurrence_indices
        )

        mean_interval = (
            recurrence_intervals.mean()
        )

        current_gap = (
            len(history_draw_sets)
            - 1
            - occurrence_indices[-1]
        )

        current_age = current_gap + 1

        cycle_proximity = math.exp(
            -0.5
            * (
                (
                    current_age
                    - mean_interval
                )
                / max(
                    0.65 * mean_interval,
                    1.0,
                )
            ) ** 2
        )

        overdue_score = min(
            current_gap / mean_interval,
            2.0,
        ) / 2.0

        raw_scores[number] = (
            0.75 * cycle_proximity
            + 0.25 * overdue_score
        )

    return normalize_scores(raw_scores)


def calculate_pair_counts(
    history_draw_sets,
    window_size=30,
):
    recent_draws = history_draw_sets[
        -window_size:
    ]

    pair_counts = Counter()

    for draw_set in recent_draws:
        for pair in combinations(
            sorted(draw_set),
            2,
        ):
            pair_counts[pair] += 1

    return pair_counts


def calculate_co_occurrence_scores(
    history_draw_sets,
):
    pair_counts = calculate_pair_counts(
        history_draw_sets,
        window_size=30,
    )

    raw_scores = {
        number: 0.0
        for number in range(1, 50)
    }

    for pair, pair_count in pair_counts.items():
        if pair_count < 2:
            continue

        first_number, second_number = pair

        pair_strength = (
            pair_count - 1
        ) ** 1.25

        raw_scores[first_number] += pair_strength
        raw_scores[second_number] += pair_strength

    return normalize_scores(raw_scores)


def calculate_mirror_scores(
    history_draw_sets,
):
    recent_20_draws = history_draw_sets[-20:]
    recent_30_draws = history_draw_sets[-30:]
    recent_50_draws = history_draw_sets[-50:]

    raw_scores = {
        number: 0.0
        for number in range(1, 50)
    }

    for first_number, second_number in mirror_pairs:
        joint_20 = sum(
            first_number in draw_set
            and second_number in draw_set
            for draw_set in recent_20_draws
        ) / len(recent_20_draws)

        joint_30 = sum(
            first_number in draw_set
            and second_number in draw_set
            for draw_set in recent_30_draws
        ) / len(recent_30_draws)

        joint_50 = sum(
            first_number in draw_set
            and second_number in draw_set
            for draw_set in recent_50_draws
        ) / len(recent_50_draws)

        weighted_joint_score = (
            0.50 * joint_20
            + 0.30 * joint_30
            + 0.20 * joint_50
        )

        first_partner_rate = np.mean([
            second_number in draw_set
            for draw_set in recent_30_draws
        ])

        second_partner_rate = np.mean([
            first_number in draw_set
            for draw_set in recent_30_draws
        ])

        raw_scores[first_number] = (
            0.70 * weighted_joint_score
            + 0.30 * first_partner_rate
        )

        raw_scores[second_number] = (
            0.70 * weighted_joint_score
            + 0.30 * second_partner_rate
        )

    return normalize_scores(raw_scores)


def calculate_same_tail_scores(
    history_draw_sets,
):
    recent_draws = history_draw_sets[-30:]
    tail_scores = {}

    for tail_digit in range(10):
        tail_numbers = {
            number
            for number in range(1, 50)
            if number % 10 == tail_digit
        }

        pair_rate = np.mean([
            len(draw_set & tail_numbers) >= 2
            for draw_set in recent_draws
        ])

        triad_rate = np.mean([
            len(draw_set & tail_numbers) >= 3
            for draw_set in recent_draws
        ])

        tail_scores[tail_digit] = (
            0.60 * pair_rate
            + 0.40 * triad_rate
        )

    raw_scores = {
        number: tail_scores[number % 10]
        for number in range(1, 50)
    }

    return normalize_scores(raw_scores)


def calculate_color_scores(
    history_draw_sets,
):
    recent_draws = history_draw_sets[-50:]

    color_counts = Counter()

    for draw_set in recent_draws:
        for number in draw_set:
            color_counts[
                get_ball_color(number)
            ] += 1

    color_sizes = {
        "紅": 17,
        "藍": 16,
        "綠": 16,
    }

    color_strength = {
        color_name: (
            color_counts[color_name]
            / color_sizes[color_name]
        )
        for color_name in color_sizes
    }

    raw_scores = {
        number: color_strength[
            get_ball_color(number)
        ]
        for number in range(1, 50)
    }

    return normalize_scores(raw_scores)

In [27]:
# 建立 DAY 綜合排名

def build_dominic_score_table(
    history_draw_sets,
):
    feature_scores = {
        "recent_score":
            calculate_recent_scores(
                history_draw_sets
            ),
        "gap_score":
            calculate_gap_scores(
                history_draw_sets
            ),
        "co_occurrence_score":
            calculate_co_occurrence_scores(
                history_draw_sets
            ),
        "mirror_score":
            calculate_mirror_scores(
                history_draw_sets
            ),
        "same_tail_score":
            calculate_same_tail_scores(
                history_draw_sets
            ),
        "color_score":
            calculate_color_scores(
                history_draw_sets
            ),
    }

    output_rows = []

    for number in range(1, 50):
        composite_score = sum(
            factor_weights[factor_name]
            * feature_scores[
                factor_name
            ][number]
            for factor_name in factor_weights
        )

        output_rows.append({
            "number": number,
            "color": get_ball_color(number),
            "recent_score":
                feature_scores[
                    "recent_score"
                ][number],
            "gap_score":
                feature_scores[
                    "gap_score"
                ][number],
            "co_occurrence_score":
                feature_scores[
                    "co_occurrence_score"
                ][number],
            "mirror_score":
                feature_scores[
                    "mirror_score"
                ][number],
            "same_tail_score":
                feature_scores[
                    "same_tail_score"
                ][number],
            "color_score":
                feature_scores[
                    "color_score"
                ][number],
            "composite_score":
                composite_score,
        })

    return pd.DataFrame(output_rows)

In [28]:
# 選擇五膽與拖腳
# 這一段加入兩項控制：
# last_draw_penalty：扣減上一期號碼的分數
# maximum_last_draw_bankers：限制五膽之中最多有多少個上一期號碼

def select_dominic_entry(
    score_df,
    history_draw_sets,
    last_draw_penalty=0.0,
    maximum_last_draw_bankers=2,
    banker_count=5,
    leg_count=20,
):
    working_df = score_df.copy()

    last_draw_set = history_draw_sets[-1]

    working_df[
        "adjusted_score"
    ] = working_df[
        "composite_score"
    ]

    working_df.loc[
        working_df["number"].isin(
            last_draw_set
        ),
        "adjusted_score",
    ] -= last_draw_penalty

    working_df = working_df.sort_values(
        [
            "adjusted_score",
            "composite_score",
            "number",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    bankers = []
    color_counts = Counter()
    last_draw_banker_count = 0

    for row in working_df.itertuples():
        number = int(row.number)
        color_name = row.color

        if (
            number in last_draw_set
            and last_draw_banker_count
            >= maximum_last_draw_bankers
        ):
            continue

        if color_counts[color_name] >= 3:
            continue

        bankers.append(number)
        color_counts[color_name] += 1

        if number in last_draw_set:
            last_draw_banker_count += 1

        if len(bankers) == banker_count:
            break

    pair_counts = calculate_pair_counts(
        history_draw_sets,
        window_size=30,
    )

    leg_rows = []

    for row in working_df.itertuples():
        number = int(row.number)

        if number in bankers:
            continue

        banker_affinity = np.mean([
            pair_counts[
                tuple(
                    sorted(
                        (number, banker)
                    )
                )
            ] / 30
            for banker in bankers
        ])

        leg_score = (
            0.75 * row.adjusted_score
            + 0.25 * banker_affinity
        )

        leg_rows.append({
            "number": number,
            "color": row.color,
            "composite_score":
                row.composite_score,
            "adjusted_score":
                row.adjusted_score,
            "banker_affinity":
                banker_affinity,
            "leg_score":
                leg_score,
        })

    leg_df = (
        pd.DataFrame(leg_rows)
        .sort_values(
            [
                "leg_score",
                "composite_score",
                "number",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .head(leg_count)
        .reset_index(drop=True)
    )

    legs = (
        leg_df["number"]
        .astype(int)
        .tolist()
    )

    banker_df = (
        working_df[
            working_df["number"].isin(
                bankers
            )
        ]
        .copy()
    )

    banker_df["banker_rank"] = (
        banker_df[
            "number"
        ].map({
            number: rank
            for rank, number
            in enumerate(
                bankers,
                start=1,
            )
        })
    )

    banker_df = banker_df.sort_values(
        "banker_rank"
    )

    return (
        bankers,
        legs,
        banker_df,
        leg_df,
    )

In [29]:
# 滾動回測參數
# 這段程式不會直接假定「上期號碼必須剔除」，而是讓歷史結果決定應否扣分。

def evaluate_parameter_setting(
    history_draw_sets,
    evaluation_indices,
    last_draw_penalty,
    maximum_last_draw_bankers,
):
    result_rows = []

    for target_index in evaluation_indices:
        training_history = history_draw_sets[
            :target_index
        ]

        actual_draw_set = history_draw_sets[
            target_index
        ]

        score_df = build_dominic_score_table(
            training_history
        )

        bankers, legs, _, _ = (
            select_dominic_entry(
                score_df=score_df,
                history_draw_sets=
                    training_history,
                last_draw_penalty=
                    last_draw_penalty,
                maximum_last_draw_bankers=
                    maximum_last_draw_bankers,
                banker_count=5,
                leg_count=20,
            )
        )

        banker_set = set(bankers)
        total_pool_set = (
            banker_set | set(legs)
        )

        banker_hits = len(
            banker_set & actual_draw_set
        )

        total_pool_hits = len(
            total_pool_set & actual_draw_set
        )

        result_rows.append({
            "target_index": target_index,
            "banker_hits": banker_hits,
            "total_pool_hits":
                total_pool_hits,
        })

    result_df = pd.DataFrame(result_rows)

    summary = {
        "last_draw_penalty":
            last_draw_penalty,
        "maximum_last_draw_bankers":
            maximum_last_draw_bankers,
        "average_banker_hits":
            result_df[
                "banker_hits"
            ].mean(),
        "banker_at_least_2_rate":
            (
                result_df[
                    "banker_hits"
                ] >= 2
            ).mean(),
        "banker_at_least_3_rate":
            (
                result_df[
                    "banker_hits"
                ] >= 3
            ).mean(),
        "average_total_pool_hits":
            result_df[
                "total_pool_hits"
            ].mean(),
        "pool_at_least_4_rate":
            (
                result_df[
                    "total_pool_hits"
                ] >= 4
            ).mean(),
        "pool_at_least_5_rate":
            (
                result_df[
                    "total_pool_hits"
                ] >= 5
            ).mean(),
    }

    summary["selection_score"] = (
        summary["average_banker_hits"]
        + 0.30
        * summary[
            "banker_at_least_2_rate"
        ]
        + 0.50
        * summary[
            "banker_at_least_3_rate"
        ]
        + 0.10
        * summary[
            "average_total_pool_hits"
        ]
    )

    return summary, result_df


def tune_dominic_parameters(
    history_draw_sets,
    tuning_draws=100,
    validation_draws=20,
):
    total_draws = len(
        history_draw_sets
    )

    tuning_start = (
        total_draws
        - validation_draws
        - tuning_draws
    )

    tuning_end = (
        total_draws
        - validation_draws
    )

    tuning_indices = range(
        tuning_start,
        tuning_end,
    )

    validation_indices = range(
        tuning_end,
        total_draws,
    )

    penalty_grid = [
        0.00,
        0.03,
        0.06,
        0.09,
        0.12,
    ]

    banker_cap_grid = [
        0,
        1,
        2,
        3,
        5,
    ]

    tuning_rows = []

    for last_draw_penalty in penalty_grid:
        for maximum_last_draw_bankers in (
            banker_cap_grid
        ):
            summary, _ = (
                evaluate_parameter_setting(
                    history_draw_sets=
                        history_draw_sets,
                    evaluation_indices=
                        tuning_indices,
                    last_draw_penalty=
                        last_draw_penalty,
                    maximum_last_draw_bankers=
                        maximum_last_draw_bankers,
                )
            )

            tuning_rows.append(summary)

    tuning_df = (
        pd.DataFrame(tuning_rows)
        .sort_values(
            [
                "selection_score",
                "average_banker_hits",
                "average_total_pool_hits",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    best_setting = tuning_df.iloc[0]

    validation_summary, validation_df = (
        evaluate_parameter_setting(
            history_draw_sets=
                history_draw_sets,
            evaluation_indices=
                validation_indices,
            last_draw_penalty=float(
                best_setting[
                    "last_draw_penalty"
                ]
            ),
            maximum_last_draw_bankers=int(
                best_setting[
                    "maximum_last_draw_bankers"
                ]
            ),
        )
    )

    return (
        tuning_df,
        validation_summary,
        validation_df,
    )

In [30]:
# 執行最新一期預測

raw_df, completed_draw_df = (
    load_completed_draws(csv_path)
)

history_draw_sets = (
    completed_draw_df[
        "draw_set"
    ].tolist()
)

print(
    "完整攪珠期數：",
    len(completed_draw_df),
)

print(
    "最後完成期數：",
    completed_draw_df.iloc[-1]["Lot"],
)

print(
    "最後完成日期：",
    completed_draw_df.iloc[-1][
        "Date"
    ].date(),
)

print(
    "最後一期七個號碼：",
    sorted(
        history_draw_sets[-1]
    ),
)


tuning_df, validation_summary, validation_df = (
    tune_dominic_parameters(
        history_draw_sets=
            history_draw_sets,
        tuning_draws=100,
        validation_draws=20,
    )
)

print("\n參數回測首10名：")

display(
    tuning_df.head(10).round(4)
)


best_last_draw_penalty = float(
    tuning_df.iloc[0][
        "last_draw_penalty"
    ]
)

best_banker_cap = int(
    tuning_df.iloc[0][
        "maximum_last_draw_bankers"
    ]
)

print(
    "\n最佳上期號碼扣分：",
    best_last_draw_penalty,
)

print(
    "五膽最多包含上期號碼：",
    best_banker_cap,
)

print("\n最近20期驗證：")

display(
    pd.DataFrame(
        [validation_summary]
    ).round(4)
)


final_score_df = (
    build_dominic_score_table(
        history_draw_sets
    )
)

bankers, legs, banker_df, leg_df = (
    select_dominic_entry(
        score_df=final_score_df,
        history_draw_sets=
            history_draw_sets,
        last_draw_penalty=
            best_last_draw_penalty,
        maximum_last_draw_bankers=
            best_banker_cap,
        banker_count=5,
        leg_count=20,
    )
)

完整攪珠期數： 929
最後完成期數： 26/101
最後完成日期： 2026-09-17
最後一期七個號碼： [4, 8, 17, 31, 41, 47, 48]

參數回測首10名：


,last_draw_penalty,maximum_last_draw_bankers,average_banker_hits,banker_at_least_2_rate,banker_at_least_3_rate,average_total_pool_hits,pool_at_least_4_rate,pool_at_least_5_rate,selection_score
0,0.12,1,0.82,0.14,0.01,3.63,0.55,0.25,1.230
1,0.12,2,0.82,0.14,0.01,3.63,0.55,0.25,1.230
2,0.12,3,0.82,0.14,0.01,3.63,0.55,0.25,1.230
3,0.12,5,0.82,0.14,0.01,3.63,0.55,0.25,1.230
4,0.06,2,0.82,0.13,0.01,3.61,0.56,0.26,1.225
5,0.06,3,0.82,0.13,0.01,3.61,0.56,0.26,1.225
6,0.06,5,0.82,0.13,0.01,3.61,0.56,0.26,1.225
7,0.06,1,0.82,0.13,0.01,3.60,0.56,0.25,1.224
8,0.12,0,0.81,0.14,0.01,3.63,0.55,0.25,1.220
9,0.06,0,0.81,0.14,0.01,3.60,0.56,0.25,1.217



最佳上期號碼扣分： 0.12
五膽最多包含上期號碼： 1

最近20期驗證：


,last_draw_penalty,maximum_last_draw_bankers,average_banker_hits,banker_at_least_2_rate,banker_at_least_3_rate,average_total_pool_hits,pool_at_least_4_rate,pool_at_least_5_rate,selection_score
0,0.12,1,0.8,0.2,0.0,3.2,0.3,0.2,1.18


In [31]:
# 顯示五膽、拖腳及20注

print("\n" + "=" * 60)
print("六合彩預測期數：", next_lot)
print("攪珠日期：", next_draw_date)
print("=" * 60)

print(
    "五膽：",
    "、".join(
        map(str, sorted(bankers))
    ),
)

print(
    "20個拖腳：",
    "、".join(
        map(str, sorted(legs))
    ),
)

print(
    "全候選池25碼：",
    "、".join(
        map(
            str,
            sorted(
                set(bankers) | set(legs)
            ),
        )
    ),
)


print("\n五膽評分：")

banker_display_columns = [
    "banker_rank",
    "number",
    "color",
    "recent_score",
    "gap_score",
    "co_occurrence_score",
    "mirror_score",
    "same_tail_score",
    "color_score",
    "composite_score",
    "adjusted_score",
]

display(
    banker_df[
        banker_display_columns
    ].round(4)
)


print("\n拖腳評分：")

display(
    leg_df[
        [
            "number",
            "color",
            "composite_score",
            "adjusted_score",
            "banker_affinity",
            "leg_score",
        ]
    ].round(4)
)


betting_rows = []

for entry_number, leg_number in enumerate(
    legs,
    start=1,
):
    entry_numbers = sorted(
        bankers + [leg_number]
    )

    betting_rows.append({
        "注項": entry_number,
        "五膽": "、".join(
            map(str, sorted(bankers))
        ),
        "拖腳": leg_number,
        "六個號碼": "、".join(
            map(str, entry_numbers)
        ),
        "每注金額": 5,
    })

betting_df = pd.DataFrame(
    betting_rows
)

print("\nHK$100部分注項方案：")

display(betting_df)

print(
    "總注數：",
    len(betting_df),
)

print(
    "總投注金額：HK$",
    betting_df["每注金額"].sum(),
)


六合彩預測期數： 26/102
攪珠日期： 2026-09-19
五膽： 7、9、23、25、34
20個拖腳： 1、11、12、14、18、19、24、26、28、29、30、31、32、33、35、38、40、42、44、45
全候選池25碼： 1、7、9、11、12、14、18、19、23、24、25、26、28、29、30、31、32、33、34、35、38、40、42、44、45

五膽評分：


,banker_rank,number,color,recent_score,gap_score,co_occurrence_score,mirror_score,same_tail_score,color_score,composite_score,adjusted_score
6,1,7,紅,0.8283,0.9891,1.0000,0.0000,1.0000,1.0000,0.8530,0.8530
22,2,23,紅,0.5193,0.6667,0.3079,0.9196,0.4286,1.0000,0.5611,0.5611
33,3,34,紅,0.6094,0.6965,0.1617,0.3424,0.6667,1.0000,0.5129,0.5129
8,4,9,藍,1.0000,0.2417,0.3468,0.0000,0.7619,0.5609,0.5070,0.5070
24,5,25,藍,0.5751,1.0000,0.0815,0.0000,0.3810,0.5609,0.4613,0.4613



拖腳評分：


,number,color,composite_score,adjusted_score,banker_affinity,leg_score
0,1,紅,0.5098,0.5098,0.0533,0.3957
1,18,紅,0.4836,0.4836,0.0400,0.3727
2,35,紅,0.4788,0.4788,0.0467,0.3708
3,12,紅,0.4573,0.4573,0.0400,0.3530
4,32,綠,0.4337,0.4337,0.0467,0.3369
5,28,綠,0.4290,0.4290,0.0333,0.3301
6,38,綠,0.4100,0.4100,0.0400,0.3175
7,45,紅,0.4046,0.4046,0.0267,0.3101
8,11,綠,0.4027,0.4027,0.0267,0.3087
9,29,紅,0.3896,0.3896,0.0600,0.3072



HK$100部分注項方案：


,注項,五膽,拖腳,六個號碼,每注金額
0,1,7、9、23、25、34,1,1、7、9、23、25、34,5
1,2,7、9、23、25、34,18,7、9、18、23、25、34,5
2,3,7、9、23、25、34,35,7、9、23、25、34、35,5
3,4,7、9、23、25、34,12,7、9、12、23、25、34,5
4,5,7、9、23、25、34,32,7、9、23、25、32、34,5
5,6,7、9、23、25、34,28,7、9、23、25、28、34,5
6,7,7、9、23、25、34,38,7、9、23、25、34、38,5
7,8,7、9、23、25、34,45,7、9、23、25、34、45,5
8,9,7、9、23、25、34,11,7、9、11、23、25、34,5
9,10,7、9、23、25、34,29,7、9、23、25、29、34,5


總注數： 20
總投注金額：HK$ 100


In [32]:
# 儲存實驗結果

output_folder = (
    "/content/drive/MyDrive/Mark_Six/"
    "prediction_results"
)

os.makedirs(
    output_folder,
    exist_ok=True,
)

banker_output_path = os.path.join(
    output_folder,
    "dominic_26_101_bankers.csv",
)

leg_output_path = os.path.join(
    output_folder,
    "dominic_26_101_legs.csv",
)

betting_output_path = os.path.join(
    output_folder,
    "dominic_26_101_betting_plan.csv",
)

tuning_output_path = os.path.join(
    output_folder,
    "dominic_26_101_parameter_test.csv",
)

banker_df.to_csv(
    banker_output_path,
    index=False,
    encoding="utf-8-sig",
)

leg_df.to_csv(
    leg_output_path,
    index=False,
    encoding="utf-8-sig",
)

betting_df.to_csv(
    betting_output_path,
    index=False,
    encoding="utf-8-sig",
)

tuning_df.to_csv(
    tuning_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("✅ 五膽結果：", banker_output_path)
print("✅ 拖腳結果：", leg_output_path)
print("✅ 投注方案：", betting_output_path)
print("✅ 參數測試：", tuning_output_path)

✅ 五膽結果： /content/drive/MyDrive/Mark_Six/prediction_results/dominic_26_101_bankers.csv
✅ 拖腳結果： /content/drive/MyDrive/Mark_Six/prediction_results/dominic_26_101_legs.csv
✅ 投注方案： /content/drive/MyDrive/Mark_Six/prediction_results/dominic_26_101_betting_plan.csv
✅ 參數測試： /content/drive/MyDrive/Mark_Six/prediction_results/dominic_26_101_parameter_test.csv
